# Lab 2: Simulating Decentralized Agreement and Control
**Course:** Blockchain and its Application | **Instructor:** Vishnu Suresh | **Term:** Zima 2026/2027  
**Student:** Kaan Yazıcıoğlu | **File:** `yazicioglu_lab2.ipynb`

---
### Goal
Simulate consensus and governance among 5 universities:
1. Quorum approval (4 of 5).
2. Crash vs. Byzantine faults.
3. Network split (Safety vs. Liveness).
4. Sybil attack vs. Permissioned whitelist.
5. Voting weight concentration & fake decentralization.
6. Final design proposal.

---
## Part 1: Trust Model and Quorum Approval
5 universities. Need 4 out of 5 approvals to accept diploma.

In [1]:
validators = [
    "University A", "University B", "University C",
    "University D", "University E",
]

proposal = {
    "student": "Kaan Yazicioglu",
    "credential": "Blockchain Fundamentals",
    "issuer": "University A",
}

def validate_proposal(proposal):
    required = {"student", "credential", "issuer"}
    missing = required - set(proposal)
    if missing:
        return False, f"Missing fields: {sorted(missing)}"
    if proposal["issuer"] not in validators:
        return False, "Issuer is not a consortium member"
    return True, "Proposal follows the rules"

def quorum_result(votes, threshold=4):
    approvals = sum(vote is True for vote in votes.values())
    responding = sum(vote is not None for vote in votes.values())
    return approvals >= threshold, approvals, responding

votes = {
    validator: validate_proposal(proposal)[0]
    for validator in validators
}

print(votes)
print(quorum_result(votes))

{'University A': True, 'University B': True, 'University C': True, 'University D': True, 'University E': True}
(True, 5, 5)


---
## Part 2: Crash and Byzantine Faults
- **Crash fault:** node dead / no answer (`None`).
- **Byzantine fault:** node lies / votes wrong.
- 1 faulty node cannot stop 4-of-5 quorum.

In [2]:
# Step 1: 1 Crashed node (University E = None)
crash_votes = votes.copy()
crash_votes["University E"] = None
print("Crash votes:", crash_votes)
print("Result:     ", quorum_result(crash_votes))
print("-> 4 honest online nodes reach quorum. System makes progress.")

Crash votes: {'University A': True, 'University B': True, 'University C': True, 'University D': True, 'University E': None}
Result:      (True, 4, 4)
-> 4 honest online nodes reach quorum. System makes progress.


In [3]:
# Step 2: 1 Dishonest node votes False on valid proposal
dishonest_votes = votes.copy()
dishonest_votes["University E"] = False
print("Dishonest votes:", dishonest_votes)
print("Result:         ", quorum_result(dishonest_votes))
print("-> 4 honest approvals still reach threshold. Bad vote ignored.")

Dishonest votes: {'University A': True, 'University B': True, 'University C': True, 'University D': True, 'University E': False}
Result:          (True, 4, 5)
-> 4 honest approvals still reach threshold. Bad vote ignored.


In [4]:
# Step 3: Bad proposal submitted (Unknown issuer)
invalid_proposal = {
    "student": "Kaan Yazicioglu",
    "credential": "Blockchain Fundamentals",
    "issuer": "Unknown University",
}

invalid_votes = {
    validator: validate_proposal(invalid_proposal)[0]
    for validator in validators
}
# Dishonest University E tries to approve fake diploma
invalid_votes["University E"] = True

print("Invalid votes:", invalid_votes)
print("Result:       ", quorum_result(invalid_votes))
print("-> Rejected! 1 dishonest vote cannot reach 4-of-5 quorum.")

Invalid votes: {'University A': False, 'University B': False, 'University C': False, 'University D': False, 'University E': True}
Result:        (False, 1, 5)
-> Rejected! 1 dishonest vote cannot reach 4-of-5 quorum.


---
## Part 3: Network Partitions (Safety vs. Liveness)
Network cuts in two: Left (3 nodes) vs. Right (2 nodes).
- **Threshold 4:** Neither side reaches 4. Chain stops (**Safety saved, Liveness lost**).
- **Threshold 3:** Split-brain danger! Traitor voting on both sides makes 2 conflicting decisions (**Safety dead**).

In [5]:
def has_quorum(voter_set, threshold):
    return len(set(voter_set)) >= threshold

left_side = {"University A", "University B", "University C"}
right_side = {"University D", "University E"}

print("Left at threshold 4: ", has_quorum(left_side, 4))
print("Right at threshold 4:", has_quorum(right_side, 4))
print("-> Neither side reaches 4. Network pauses. Prevents double-spend/split-brain.")

Left at threshold 4:  False
Right at threshold 4: False
-> Neither side reaches 4. Network pauses. Prevents double-spend/split-brain.


In [6]:
# Lower threshold to 3 + traitor E votes on both sides (equivocation)
proposal_x_voters = {"University A", "University B", "University E"}
proposal_y_voters = {"University C", "University D", "University E"}

print("X at threshold 3:", has_quorum(proposal_x_voters, 3))
print("Y at threshold 3:", has_quorum(proposal_y_voters, 3))
print("Voted for both:  ", proposal_x_voters & proposal_y_voters)
print("-> Disaster! Both conflicting proposals finalized at same time.")

X at threshold 3: True
Y at threshold 3: True
Voted for both:   {'University E'}
-> Disaster! Both conflicting proposals finalized at same time.


### Comparison: Threshold 4 vs. Threshold 3

| Threshold | 3-2 Network Partition | Conflicting Quorum Risk |
| :--- | :--- | :--- |
| **4 of 5** | Neither side finalizes (Liveness pauses) | Zero: One traitor cannot finalize two sides |
| **3 of 5** | Left side finalizes (Liveness continues) | High: Traitor voting twice approves 2 conflicting histories |

---
## Part 4: Sybil Resistance
Open voting without ID check: attacker creates 20 fake bots and outvotes 5 real universities.
Defense: Permissioned membership list (or PoW/PoS in open networks).

In [7]:
def simple_majority(votes):
    approvals = sum(votes.values())
    return approvals > len(votes) / 2

honest_votes = {validator: False for validator in validators}
fake_votes = {f"Fake identity {number}": True for number in range(1, 21)}
open_votes = {**honest_votes, **fake_votes}

print("Total voters: ", len(open_votes))
print("Attacker wins:", simple_majority(open_votes))
print("-> Sybil attack works! Cheap fake identities steal control.")

Total voters:  25
Attacker wins: True
-> Sybil attack works! Cheap fake identities steal control.


In [8]:
# Fix: Whitelist authorized validators
authorized = set(validators)
authorized_votes = {
    identity: vote
    for identity, vote in open_votes.items()
    if identity in authorized
}

print("Authorized voters counted:", len(authorized_votes))
print("Attacker wins:            ", simple_majority(authorized_votes))
print("-> Fake identities blocked by membership check.")

Authorized voters counted: 5
Attacker wins:             False
-> Fake identities blocked by membership check.


### Caveman Summary: Sybil Defense
- **Consortium:** Whitelist of known cryptographic keys.
- **Proof of Work (Bitcoin):** 1 CPU/hashrate = 1 vote. Identities are free, electricity is not.
- **Proof of Stake (Ethereum):** 1 coin = 1 vote. Identities are free, capital is not.

---
## Part 5: Effective Control and Unequal Voting Power
Node count is deceptive.
If Validator A holds 70% of voting weight, having 5 nodes on paper means nothing. Validator A is dictator.

In [9]:
def minimum_coalition(weights, threshold):
    ordered = sorted(weights.items(), key=lambda item: item[1], reverse=True)
    coalition = []
    total = 0
    for validator, weight in ordered:
        coalition.append(validator)
        total += weight
        if total >= threshold:
            return coalition, total
    return coalition, total

equal_weights = {"A": 20, "B": 20, "C": 20, "D": 20, "E": 20}
concentrated_weights = {"A": 40, "B": 25, "C": 15, "D": 10, "E": 10}

print("Equal (threshold 67):       ", minimum_coalition(equal_weights, 67))
print("Concentrated (threshold 67):", minimum_coalition(concentrated_weights, 67))

Equal (threshold 67):        (['A', 'B', 'C', 'D'], 80)
Concentrated (threshold 67): (['A', 'B', 'C'], 80)


In [10]:
# Step 3: Extreme case (Validator A has 70%, remaining 30% split)
extreme_weights = {"A": 70, "B": 10, "C": 10, "D": 5, "E": 5}
res_coalition, res_weight = minimum_coalition(extreme_weights, 67)

print(f"Extreme case (threshold 67): Coalition={res_coalition}, Weight={res_weight}")
print("-> Coalition size = 1! Validator A decides everything alone.")
print("-> Caveman answer: Validator count does NOT measure decentralization. Weight distribution does.")

Extreme case (threshold 67): Coalition=['A'], Weight=70
-> Coalition size = 1! Validator A decides everything alone.
-> Caveman answer: Validator count does NOT measure decentralization. Weight distribution does.


---
## Part 6: Decentralization Design Proposal (Final Task)

### Selected Scenario:
**Scenario 1: Five Universities Sharing Micro-Credentials (Permissioned Consortium)**

#### 1. Who submits records and who validates them?
- **Submitters:** Only accredited member universities submit credential issuance/revocation proposals.
- **Validators:** All 5 university validator nodes validate proposals against shared schema and issuer cryptographic signatures.

#### 2. Who may read the ledger and who governs membership or upgrades?
- **Readers:** Public access (students, employers, verifiers) via open read-only RPC/API to verify diploma authenticity.
- **Governance:** Consortium board (5 university representatives) with 4-of-5 threshold multisig to add/remove members or upgrade smart contracts.

#### 3. Which failures or collusion the design must tolerate?
- Tolerates **1 crash fault** ($N=5, T=4 
ightarrow$ system continues with 4 nodes).
- Tolerates **1 Byzantine (dishonest) fault** ($T=4$ prevents a rogue node from approving fake credentials or blocking valid ones).
- Requires at least 2 malicious universities colluding to cause damage.

#### 4. How the design resists fake identities (Sybil attacks)?
- **Permissioned PKI whitelist:** Node identities are tied to authenticated cryptographic certificates (e.g. X.509). Unregistered keys cannot vote. No wasteful PoW/PoS needed.

#### 5. What quorum, fork-choice, or finality rule determines acceptance?
- **Rule:** Deterministic 4-of-5 ($80\%$) quorum rule.
- **Finality:** Instant deterministic finality (BFT-style). Once 4 signatures are collected, the block is permanent. No forks or chain reorgs allowed.

#### 6. Where effective control could become concentrated?
- **Hosting / Cloud dependency:** If all universities host nodes on AWS us-east-1.
- **Software monoculture:** All nodes running identical unpatched client software.
- **Unequal voting:** If voting weight scales with university student count rather than strict 1-institution = 1-vote.

#### 7. Whether a simpler signed database would meet the same need?
- **Why database fails:** A single database has a single administrator. One university can secretly modify records, delete competitor credentials, or suffer a single-point-of-failure outage.
- **Why blockchain succeeds:** Provides mutual distrust protection. No single university can unilaterally rewrite or censor history.